# UUID Package (npm)

**Tags:** #nodejs #npm #javascript #utilities

The [`uuid`](https://www.npmjs.com/package/uuid) package is the most widely used, zero-dependency utility for generating RFC 9562 (formerly RFC 4122) compliant Universally Unique Identifiers in JavaScript and Node.js.

---

## Why UUIDs at all?

An auto-incrementing integer ID requires a central authority — the database — to hand out the next value. A UUID can be generated anywhere: in the browser, in a worker, on three servers at once, offline. Nobody has to coordinate.

That buys you:

- **Client-side ID generation** — create the record's ID before it ever hits the server
- **Merge-safe data** — two datasets from different sources won't have colliding keys
- **No enumeration leak** — `/users/1247` tells an attacker you have ~1247 users; a UUID tells them nothing

The cost is size (36 chars vs 4 bytes) and, for v4, index fragmentation. See [[#UUIDs as database keys]] below.

---

## Quick start

Install:

```bash
npm install uuid
```

Generate a random UUIDv4 — the most common version:

```javascript
import { v4 as uuidv4 } from 'uuid';

const id = uuidv4();
console.log(id); // '1b9d6bcd-bbfd-4b2d-9b5d-ab8dfbbd4bed'
```

CommonJS equivalent:

```javascript
const { v4: uuidv4 } = require('uuid');
```

> [!note] Import the specific version, not the whole package
> `import { v4 as uuidv4 } from 'uuid'` lets bundlers tree-shake away the versions you don't use. `import * as uuid from 'uuid'` pulls in everything.

---

## Supported versions

| Version | Basis | Use when |
|---|---|---|
| `v1()` | Timestamp + MAC address | You need traceable creation time (leaks the MAC) |
| `v3()` | MD5 hash of namespace + name | Deterministic IDs, legacy |
| `v4()` | Cryptographically random | Default choice — security, privacy, no coordination |
| `v5()` | SHA-1 hash of namespace + name | Deterministic IDs, preferred over v3 |
| `v6()` | Reordered v1 | v1 semantics but sortable |
| `v7()` | Unix timestamp + random | **Database primary keys** — sortable and non-leaky |

### Deterministic IDs (v5)

v3 and v5 are *not* random. The same namespace and name always produce the same UUID:

```javascript
import { v5 as uuidv5 } from 'uuid';

const MY_NAMESPACE = '1b671a64-40d5-491e-99b0-da01ff1f3341';

uuidv5('hello', MY_NAMESPACE); // always the same output
```

Useful for generating a stable ID from something you already have — a URL, an email, an external system's key — without storing a mapping table.

### v7 — the one to reach for in databases

```javascript
import { v7 as uuidv7 } from 'uuid';

uuidv7(); // '01936d4f-6e2a-7c3d-8f1e-2b4a6c8d0e2f'
```

The leading bits are a millisecond timestamp, so v7 UUIDs sort chronologically as strings. This fixes the biggest practical problem with v4 as a primary key.

---

## Utility methods

```javascript
import { validate, version, parse, stringify, NIL, MAX } from 'uuid';

validate('not-a-uuid');   // false
version('1b9d6bcd-...');  // 4
parse('1b9d6bcd-...');    // Uint8Array(16) — 16 raw bytes
stringify(bytes);         // back to the dashed string form

NIL; // '00000000-0000-0000-0000-000000000000'
MAX; // 'ffffffff-ffff-ffff-ffff-ffffffffffff'
```

`parse` / `stringify` matter when you're storing UUIDs as `BINARY(16)` rather than `CHAR(36)` — see below.

---

## TypeScript

**As of `uuid` v9.0.0, types ship with the package.** You do not need `@types/uuid`; it's now a stub that exists only for backwards compatibility. Older guides still tell you to install it — ignore them unless you're pinned to v8 or earlier.

```bash
# Only if stuck on uuid@8 or below
npm install --save-dev @types/uuid
```

---

## Native alternative — no package required

Modern runtimes have this built in:

```javascript
const id = crypto.randomUUID();
```

Available in Node 14.17+ (as `require('crypto').randomUUID`) and globally from Node 19+, plus all current browsers.

> [!warning] Browser caveat
> `crypto.randomUUID()` requires a **secure context** — HTTPS or `localhost`. It's `undefined` on a plain-HTTP page, which is a nasty surprise on a staging server. Node has no such restriction.

**Rule of thumb:** if you only need v4 and control your runtime, use the native API. Reach for the package when you need v5's determinism, v7's sortability, or the parse/validate helpers.

---

## UUIDs as database keys

Worth knowing if you're modelling tables rather than just generating IDs:

**Storage.** A UUID is 128 bits. Stored as `CHAR(36)` it costs 36 bytes and every index entry carries that weight. Stored as `BINARY(16)` (MySQL) or the native `uniqueidentifier` (SQL Server) / `uuid` (Postgres) type, it costs 16. On a table with several UUID foreign keys and a few million rows, that difference is real.

**Index fragmentation.** v4 UUIDs are random, so inserts land at random points in a clustered index — page splits, poor cache locality, bloated indexes. This is the classic argument against UUID primary keys and it's a legitimate one.

**v7 mostly solves it.** Because v7 is timestamp-prefixed, sequential inserts land at the end of the index, the same as an identity column. If you want UUID keys in a real OLTP table, use v7, not v4.

**For analytics/BI specifically:** UUID join keys are noticeably slower than integer keys on large joins, and they're miserable to eyeball during debugging. A common pattern is to keep the UUID as the external/public identifier and use a surrogate integer key internally for joins — you get stable public IDs without paying the join cost on every query.

---

## Collision probability

For v4: 122 random bits, so roughly 5.3×10³⁶ possible values. You would need to generate about 2.7×10¹⁸ UUIDs before hitting a 50% chance of a single collision.

In practice: don't write collision-handling logic for v4. Do make sure your source of randomness is actually cryptographic — the package uses `crypto.getRandomValues()` under the hood, which is why hand-rolled `Math.random()` UUID snippets found on Stack Overflow are a bad idea.

---

## References

- [uuid on npm](https://www.npmjs.com/package/uuid)
- [uuidjs/uuid on GitHub](https://github.com/uuidjs/uuid)
- [RFC 9562 — UUID specification](https://datatracker.ietf.org/doc/html/rfc9562)